In [1]:
import torch
import torch.nn as nn
import torch.nn.functional as F
from datasets import load_dataset
from transformers import AutoTokenizer

In [3]:
device = "cuda" if torch.cuda.is_available() else "cpu"

# ---- load dataset -----

dataset = load_dataset("wikitext","wikitext-2-raw-v1")
text ="\n".join(dataset["train"]["text"][:500])

/usr/local/lib/python3.12/dist-packages/huggingface_hub/utils/_auth.py:94: UserWarning: 
The secret `HF_TOKEN` does not exist in your Colab secrets.
To authenticate with the Hugging Face Hub, create a token in your settings tab (https://huggingface.co/settings/tokens), set it as secret in your Google Colab and restart your session.
You will be able to reuse this secret in all of your notebooks.
Please note that authentication is recommended but still optional to access public models or datasets.
  warnings.warn(


README.md: 0.00B [00:00, ?B/s]

wikitext-2-raw-v1/test-00000-of-00001.pa(…):   0%|          | 0.00/733k [00:00<?, ?B/s]

wikitext-2-raw-v1/train-00000-of-00001.p(…):   0%|          | 0.00/6.36M [00:00<?, ?B/s]

wikitext-2-raw-v1/validation-00000-of-00(…):   0%|          | 0.00/657k [00:00<?, ?B/s]

Generating test split:   0%|          | 0/4358 [00:00<?, ? examples/s]

Generating train split:   0%|          | 0/36718 [00:00<?, ? examples/s]

Generating validation split:   0%|          | 0/3760 [00:00<?, ? examples/s]

In [4]:
# ------load GPT2 Tokenizer------

tokenizer = AutoTokenizer.from_pretrained("gpt2")
tokens = tokenizer.encode(text)

data = torch.tensor(tokens , dtype=torch.long)
vocab_size = tokenizer.vocab_size


config.json:   0%|          | 0.00/665 [00:00<?, ?B/s]

tokenizer_config.json:   0%|          | 0.00/26.0 [00:00<?, ?B/s]

vocab.json: 0.00B [00:00, ?B/s]

merges.txt: 0.00B [00:00, ?B/s]

tokenizer.json: 0.00B [00:00, ?B/s]

Token indices sequence length is longer than the specified maximum sequence length for this model (26186 > 1024). Running this sequence through the model will result in indexing errors


In [5]:
block_size = 32
batch_size = 32
embed_size = 128
num_heads = 4
epochs = 300
lr=3e-4

In [16]:
def get_batch():
    ix = torch.randint(len(data) - block_size, (batch_size,))

    x = torch.stack([data[i:i+block_size] for i in ix])
    y = torch.stack([data[i+1:i+block_size+1] for i in ix])

    return x.to(device), y.to(device)

In [24]:
class transformerBlock(nn.Module):
    def __init__(self, embed_size, num_heads):
        super().__init__()
        self.attn = nn.MultiheadAttention(
            embed_size,
            num_heads,
            batch_first=True
        )
        self.ff = nn.Sequential(
            nn.Linear(embed_size, 4 * embed_size),
            nn.ReLU(),
            nn.Linear(4 * embed_size, embed_size)
        )
        self.ln1 = nn.LayerNorm(embed_size)
        self.ln2 = nn.LayerNorm(embed_size)

    def forward(self, x, mask):
        attn_out, _ = self.attn(x, x, x, attn_mask=mask)
        x = self.ln1(x + attn_out)
        ff_out = self.ff(x)
        x = self.ln2(x + ff_out)
        return x

In [28]:
class MiniGPT(nn.Module):
  def __init__(self):
    super().__init__()
    self.token_embed = nn.Embedding(vocab_size,embed_size)
    self.pos_embed = nn.Embedding(block_size,embed_size)
    self.block = transformerBlock(embed_size,num_heads)
    self.ln = nn.LayerNorm(embed_size)
    self.head = nn.Linear(embed_size,vocab_size)

  def forward(self,x):
    B,T = x.shape
    token = self.token_embed(x)
    pos = self.pos_embed(torch.arange(T ,device=device))
    x = token + pos
    mask = torch .triu(torch.ones(T,T)*float('-inf'),diagonal = 1).to(device)
    x = self.block(x,mask)
    x = self.ln(x)
    logits = self.head(x)
    return logits
model = MiniGPT().to(device)
optimizer = torch.optim.Adam(model.parameters(),lr=lr)

In [30]:
for epoch in range(epochs):
  xb,yb = get_batch()
  logits = model(xb)
  loss = F.cross_entropy(logits.view(-1,vocab_size),yb.view(-1))
  optimizer.zero_grad()
  loss.backward()
  optimizer.step()
  if epoch % 10 == 0:
    print(f"Epoch {epoch}, Loss: {loss.item()}")

Epoch 0, Loss: 6.084943771362305
Epoch 10, Loss: 6.139791488647461
Epoch 20, Loss: 5.907607555389404
Epoch 30, Loss: 5.989809989929199
Epoch 40, Loss: 6.047354698181152
Epoch 50, Loss: 5.76108980178833
Epoch 60, Loss: 5.864473342895508
Epoch 70, Loss: 6.000709533691406
Epoch 80, Loss: 5.963089466094971
Epoch 90, Loss: 5.752289295196533
Epoch 100, Loss: 6.029407501220703
Epoch 110, Loss: 5.836113929748535
Epoch 120, Loss: 5.820700168609619
Epoch 130, Loss: 5.70258903503418
Epoch 140, Loss: 5.831729888916016
Epoch 150, Loss: 5.234031677246094
Epoch 160, Loss: 5.762493133544922
Epoch 170, Loss: 5.454023361206055
Epoch 180, Loss: 5.372375011444092
Epoch 190, Loss: 5.58139181137085
Epoch 200, Loss: 5.584559440612793
Epoch 210, Loss: 5.59696102142334
Epoch 220, Loss: 5.2785844802856445
Epoch 230, Loss: 5.341772079467773
Epoch 240, Loss: 5.22902774810791
Epoch 250, Loss: 5.522036552429199
Epoch 260, Loss: 5.258072853088379
Epoch 270, Loss: 5.401390552520752
Epoch 280, Loss: 5.14121150970459
E

In [33]:
def generate(model, length=100, temperature=0.8):
    model.eval()
    tokens = tokenizer.encode(start_text)
    tokens = torch.tensor(tokens).unsqueeze(0).to(device)
    for _ in range(length):
        tokens_cond = tokens[:, -block_size:]
        logits = model(tokens_cond)
        logits = logits[:, -1, :] / temperature
        probs = F.softmax(logits, dim=-1)
        next_token = torch.argmax(probs, dim=-1).unsqueeze(1)
        tokens = torch.cat((tokens, next_token), dim=1)
    output = tokenizer.decode(tokens[0].tolist())
    return output

In [37]:
start_text = "Once upon a time"

print(generate(model))


Once upon a time . The Gambia 's Got to the Blue Jackets , the Blue Jackets 's Got to the Blue Jackets , and the game 's Got to the Blue Jackets , the Blue Jackets , the Blue Jackets , the Blue Jackets , the Blue Jackets , the Blue Jackets , the Blue Jackets 's Got to the Blue Jackets , the Blue Jackets , the Blue Jackets , the Blue Jackets , the Blue Jackets , the Blue Jackets , the Blue Jackets 's Got to the Blue Jackets , the Blue Jackets ,
